# Aula 02 — Contagem e combinatória para probabilidade

## Objetivo

Validar fórmulas de contagem por enumeração em casos pequenos, calcular uma probabilidade combinatória exata e comparar esse resultado com uma simulação reproduzível.

> Este notebook complementa a Aula 02 do módulo **Probabilidade, Estatística e Teoria da Informação**.

## Premissas e limites

- Objetos comparados em uma mesma enumeração são distintos, exceto quando indicado.
- A amostra do lote é uniforme entre os subconjuntos de quatro itens e ocorre sem reposição.
- A seed fixa torna a execução reproduzível no ambiente informado, mas não é uma hipótese probabilística.
- Enumeração serve como teste para casos pequenos; não deve materializar espaços combinatórios enormes.
- A simulação aproxima a probabilidade exata e não substitui a demonstração combinatória.

## 1. Preparação

O Google Colab já inclui NumPy e Matplotlib. Em ambiente local:

```bash
python -m pip install "numpy>=1.24" "matplotlib>=3.7" jupyter
```

In [ ]:
import sys
from itertools import combinations, permutations, product
from math import comb, factorial, perm

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seed:", SEED)

## 2. Fórmulas exatas e identidades

`math.factorial`, `math.perm` e `math.comb` retornam inteiros exatos. Os testes abaixo conferem casos-base, simetria e a identidade que soma subconjuntos de todos os tamanhos.

In [ ]:
resultados = {
    "0!": factorial(0),
    "5!": factorial(5),
    "P(10, 3)": perm(10, 3),
    "C(10, 3)": comb(10, 3),
    "C(20, 4)": comb(20, 4),
    "subconjuntos de 10 itens": sum(comb(10, k) for k in range(11)),
}

for nome, valor in resultados.items():
    print(f"{nome:27} = {valor:,}".replace(",", "."))

assert factorial(0) == 1
assert perm(10, 3) == 720
assert comb(10, 3) == comb(10, 7) == 120
assert sum(comb(10, k) for k in range(11)) == 2**10
print("Identidades verificadas.")

## 3. Enumeração valida a fórmula em escala pequena

Com quatro símbolos, podemos materializar os resultados e comparar seus tamanhos com as fórmulas. Não faça isso com um vocabulário real ou muitas posições.

In [ ]:
itens = tuple("ABCD")

com_repeticao = list(product(itens, repeat=2))
ordenados_sem_repeticao = list(permutations(itens, 2))
nao_ordenados_sem_repeticao = list(combinations(itens, 2))

print("Sequências com repetição:", com_repeticao)
print("Permutações sem repetição:", ordenados_sem_repeticao)
print("Combinações:", nao_ordenados_sem_repeticao)

assert len(com_repeticao) == 4**2 == 16
assert len(ordenados_sem_repeticao) == perm(4, 2) == 12
assert len(nao_ordenados_sem_repeticao) == comb(4, 2) == 6
print("Enumeração e fórmulas concordam.")

## 4. Ordenações com elementos repetidos

Na palavra `DADO`, trocar os dois `D` entre si não cria uma palavra nova. Enumeramos as permutações das posições e removemos duplicatas para validar $4!/2!=12$.

In [ ]:
palavra = "DADO"
ordenacoes_distintas = sorted({"".join(p) for p in permutations(palavra)})
formula_dado = factorial(4) // factorial(2)

print("Ordenações distintas:", ordenacoes_distintas)
print("Total:", len(ordenacoes_distintas))

assert len(ordenacoes_distintas) == formula_dado == 12

## 5. Probabilidade exata: dois defeituosos em quatro itens

O lote possui 5 itens defeituosos e 15 adequados. Uma amostra uniforme escolhe 4 dos 20 itens sem reposição. Os casos favoráveis escolhem 2 itens de cada grupo.

$$
P(X=2)=rac{inom52inom{15}{2}}{inom{20}{4}}.
$$

In [ ]:
total = comb(20, 4)
favoraveis = comb(5, 2) * comb(15, 2)
p_exata = favoraveis / total

print(f"Amostras possíveis: {total:,}".replace(",", "."))
print(f"Amostras favoráveis: {favoraveis:,}".replace(",", "."))
print(f"Probabilidade exata: {p_exata:.6f} ({100*p_exata:.2f}%)")

assert total == 4_845
assert favoraveis == 1_050
assert np.isclose(p_exata, 70 / 323)

## 6. Simulação reproduzível

Para cada repetição, geramos uma pontuação aleatória contínua para cada um dos 20 itens e selecionamos as quatro menores. Como não há empates com probabilidade positiva no modelo contínuo, cada subconjunto de quatro índices é selecionado uniformemente. Os índices `0` a `4` representam os defeituosos.

In [ ]:
N_SIMULACOES = 100_000
TAMANHO_LOTE = 20
TAMANHO_AMOSTRA = 4
N_DEFEITUOSOS = 5

pontuacoes = rng.random((N_SIMULACOES, TAMANHO_LOTE))
amostras = np.argpartition(pontuacoes, TAMANHO_AMOSTRA - 1, axis=1)[:, :TAMANHO_AMOSTRA]
qtd_defeituosos = (amostras < N_DEFEITUOSOS).sum(axis=1)
p_simulada = np.mean(qtd_defeituosos == 2)
erro_absoluto = abs(p_simulada - p_exata)

print(f"Probabilidade exata:    {p_exata:.6f}")
print(f"Frequência simulada:    {p_simulada:.6f}")
print(f"Erro absoluto:          {erro_absoluto:.6f}")

assert erro_absoluto < 0.005
print("Simulação compatível com o valor exato dentro da tolerância declarada.")

## 7. Visualização da explosão combinatória

O gráfico compara o número de subconjuntos não vazios de atributos com o número de grades de busca formadas por três hiperparâmetros, cada um com cinco opções. A escala vertical é logarítmica.

In [ ]:
dimensoes = np.array([5, 10, 15, 20, 25, 30, 40, 50])
subconjuntos = np.array([2**int(d) - 1 for d in dimensoes], dtype=np.float64)
grade_cinco_opcoes = np.array([5**int(d) for d in dimensoes], dtype=np.float64)

fig, ax = plt.subplots(figsize=(10, 5.2))
ax.plot(dimensoes, subconjuntos, marker="o", linewidth=2,
        color="#16b8f3", label=r"subconjuntos não vazios: $2^d-1$")
ax.plot(dimensoes, grade_cinco_opcoes, marker="s", linewidth=2,
        color="#ffb347", label=r"grade com 5 opções: $5^d$")
ax.set_yscale("log")
ax.set_xlabel("número de atributos ou decisões (d)")
ax.set_ylabel("número de configurações — escala log")
ax.set_title("Explosão combinatória")
ax.grid(alpha=0.25, which="both")
ax.legend()
plt.show()

print(f"30 atributos → {2**30 - 1:,} subconjuntos não vazios".replace(",", "."))
assert 2**30 - 1 == 1_073_741_823

## 8. Desafios

1. Troque `itens` por cinco símbolos e confirme $5^2$, $P(5,2)$ e $inom52$.
2. Valide por enumeração que `BANANA` possui $6!/(3!2!)=60$ ordenações distintas.
3. Mude o lote para 30 itens, 6 defeituosos e amostra de 5; calcule e simule a probabilidade de exatamente um defeituoso.
4. Compare $2^d-1$ para $d=20$, $30$ e $50$. Em qual ponto a enumeração deixa de ser razoável em seu computador?
5. Crie uma função que receba `n`, `k`, `ordem_importa` e `repeticao` e devolva a contagem apropriada, validando entradas inválidas.
6. Escreva uma frase sobre o que a contagem prova e outra sobre o que ela não permite concluir sem equiprobabilidade.

## Conclusões

- Casos alternativos disjuntos são somados; escolhas em etapas são multiplicadas.
- Ordem e repetição definem o objeto matemático antes da fórmula.
- Combinações removem as $k!$ ordens internas de cada grupo.
- Contagem vira probabilidade por $|A|/|\Omega|$ apenas em espaços finitos equiprováveis.
- Enumeração é um bom teste para exemplos pequenos; fórmulas evitam materializar espaços enormes.
- Espaços de atributos, hiperparâmetros e sequências crescem exponencialmente.

## Próxima etapa

Siga para a Aula 03 — **Probabilidade condicional e independência**. Nela, a informação observada restringirá o universo de referência.